In [ ]:
import sys
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import ttest_ind, pearsonr
from adjustText import adjust_text

sys.path.append(str(Path.cwd().parent / "scripts"))
from style import set_default_style

set_default_style()

In [ ]:
## read in data
bulk_lipidomics = pd.read_csv('../data/lipogrid/pilot/bulk_lipidomics/BulkLipidomics_quant_species_nmol_mgDNA.tsv', sep = '\t', index_col = 0)

## replace missing values with 0
bulk_lipidomics = bulk_lipidomics.fillna(0)
bulk_lipidomics

In [ ]:
## group gene columns based on the first word before the underscore
gene_groups = {}
for col in bulk_lipidomics.columns:
    group = col.split('_')[0]  # Extract the first word before '_'
    if group not in gene_groups:
        gene_groups[group] = []
    gene_groups[group].append(col)
gene_groups

## only keep rows in bulk_lipidomics for which there is at least one gene group with values for all samples in that group and above a noise threshold (e.g. 0.2)
noise_threshold = 0.2
filtered_bulk_lipidomics = bulk_lipidomics.copy()
for index, row in bulk_lipidomics.iterrows():
    keep_row = False
    for group, columns in gene_groups.items():
        if (row[columns] > noise_threshold).all():  # Check if all values in the group are above the noise threshold
            keep_row = True
            break
    if not keep_row:
        filtered_bulk_lipidomics.drop(index, inplace=True)
filtered_bulk_lipidomics

In [ ]:
# Bulk lipidomics reports both fatty-acid tails separately (e.g. PC 14:0_14:0), which
# isn't resolved in MALDI-MSI data. Collapse each species to its summed composition
# (PC 14:0_14:0 -> PC 28:0, PC 16:0_18:2 -> PC 34:2) so the two datasets are comparable.
# Handles the O-/P- ether-lipid prefix and the ;O2 sphingoid-base suffix too.
def collapse_lipid_species(lipid_name):
    match = re.match(r'([A-Za-z]+)\s(O-|P-)?(\d+):(\d+)(;O2)?_(O-|P-)?(\d+):(\d+)', lipid_name)
    if match:
        lipid_class = match.group(1)
        prefix1 = match.group(2) if match.group(2) else ''
        prefix2 = match.group(6) if match.group(6) else ''
        suffix = match.group(5) if match.group(5) else ''
        total_carbons = int(match.group(3)) + int(match.group(7))
        total_double_bonds = int(match.group(4)) + int(match.group(8))
        return f'{lipid_class} {prefix1 if prefix1 or prefix2 else ""}{total_carbons}:{total_double_bonds}{suffix if suffix else ""}'
    else:
        return lipid_name  # Return the original name if it doesn't match the pattern

collapsed_bulk_lipidomics = filtered_bulk_lipidomics.copy()
collapsed_bulk_lipidomics.index = collapsed_bulk_lipidomics.index.map(collapse_lipid_species)
collapsed_bulk_lipidomics = collapsed_bulk_lipidomics.groupby(collapsed_bulk_lipidomics.index).sum()
collapsed_bulk_lipidomics

In [ ]:
## Group columns by the first word before '_' and calculate log2 fold change for each group, then perform t-tests for each lipid species
bulk_lipidomics_log2fc = collapsed_bulk_lipidomics.copy()

# Identify control and gene columns
control_cols = [col for col in bulk_lipidomics_log2fc.columns if 'control' in col]
gene_cols = [col for col in bulk_lipidomics_log2fc.columns if 'control' not in col]

# Group gene columns by the first word before '_'
gene_groups = {}
for col in gene_cols:
    group = col.split('_')[0]  # Extract the first word before '_'
    if group not in gene_groups:
        gene_groups[group] = []
    gene_groups[group].append(col)

# Calculate the mean of control columns
control_mean = bulk_lipidomics_log2fc[control_cols].mean(axis=1)

# Calculate log2 fold change for each gene group and perform t-tests for each lipid species
ttest_results = {}
for group, cols in gene_groups.items():
    group_mean = bulk_lipidomics_log2fc[cols].mean(axis=1)  # Mean of the group columns
    log2fc = np.log2((group_mean + 0.1) / (control_mean + 0.1))  # Log2 fold change
    bulk_lipidomics_log2fc[f'{group}_log2FC'] = log2fc

    # Perform t-test for each lipid species
    p_values = []
    for lipid in collapsed_bulk_lipidomics.index:
        group_values = collapsed_bulk_lipidomics.loc[lipid, cols].dropna()
        control_values = collapsed_bulk_lipidomics.loc[lipid, control_cols].dropna()

        if len(group_values) > 0 and len(control_values) > 0:  # Ensure non-empty arrays
            t_stat, p_value = ttest_ind(group_values, control_values)
        else:
            p_value = np.nan  # Assign NaN if data is invalid

        p_values.append(p_value)

    # Store p-values for each lipid species in a separate column
    bulk_lipidomics_log2fc[f'{group}_pvalue'] = p_values

# Display the updated DataFrame
bulk_lipidomics_log2fc

In [ ]:
## cap lowest pval to 10^-10 using a new dataframe as to not overwrite the original one
target = 'LPCAT3'  # <-- change this to the gene you want to plot

all_results_df_capped = bulk_lipidomics_log2fc.copy()
all_results_df_capped[f'{target}_pvalue'] = all_results_df_capped[f'{target}_pvalue'].apply(lambda x: max(x, 10**-10))

set_default_style(font_size=28)

plt.figure(figsize=(12, 12))

texts = []
plt.scatter(all_results_df_capped[f'{target}_log2FC'], -np.log10(all_results_df_capped[f'{target}_pvalue']), color='black', alpha=0.6, s=12)

sig_mask = (all_results_df_capped[f'{target}_pvalue'] < 0.05) & (abs(all_results_df_capped[f'{target}_log2FC']) > 1)
for lipid, row in all_results_df_capped[sig_mask].iterrows():
    texts.append(
        plt.text(row[f'{target}_log2FC'], -np.log10(row[f'{target}_pvalue']), lipid, fontsize=20, color='blue', ha='center', va='center')
    )

plt.axhline(y=-np.log10(0.05), color='red', linestyle='--', label='p-value = 0.05')
plt.axvline(x=1, color='black', linestyle='--', label='Fold Change = 1')
plt.axvline(x=-1, color='black', linestyle='--', label='Fold Change = -1')
plt.title(f'Effect {target} knockout')
plt.xlabel(f'Log2FC ({target} shRNA / control shRNA)')
plt.ylabel('-Log10 p-value')
plt.grid()
plt.tight_layout()

adjust_text(
    texts,
    arrowprops=dict(arrowstyle='-', color='gray', lw=1),
    expand_text=(1.3, 1.3),
    expand_points=(1.2, 1.2),
    force_text=0.3,
    force_points=0.3,
    force_pull=7,
    time_lim=10,
    min_arrow_len=15,
)

plt.savefig(f'../data/lipogrid/pilot/analysis/final_4_runs/volcano_plot_bulk_{target}_log2FC_pvalue.pdf')
plt.show()

In [ ]:
## import the calculated log2 fold change and p-values from the LipoGrid experiments
LipoGrid_log2FC_df = pd.read_csv('../data/lipogrid/pilot/analysis/final_4_runs/all_lipid_log2FC_pvalues_per_gene.csv', index_col=0)
# add to each column "_LG"
LipoGrid_log2FC_df.columns = [f"{col}_LG" for col in LipoGrid_log2FC_df.columns]
## make one dataframe with LipoGrid_log2FC_df and bulk_lipidomics_log2fc only keeping the rows where both indexes are present
## unfortunately several lipids found in LipoGrid are not found in bulk lipidomics, and vice versa, so the number of lipid species that can be compared is limited to 97
combined_df = pd.merge(bulk_lipidomics_log2fc, LipoGrid_log2FC_df, left_index=True, right_index=True, how='inner')
combined_df

In [ ]:
## Plot for all unique targets in KO_genes
KO_genes = bulk_lipidomics_log2fc.columns.str.split('_').str[0].unique()
## Remove control1 and control2
KO_genes = [gene for gene in KO_genes if gene not in ['control1', 'control2']]
KO_genes

In [ ]:
## clip outliers in the combined_df for plotting (PS38:4 has -3.479422275616293 in bulk, will add to plot))
## clip columns with log2FC in their name
log2fc_columns = [col for col in combined_df.columns if 'log2FC' in col]
combined_df_clipped = combined_df.copy()
combined_df_clipped[log2fc_columns] = combined_df_clipped[log2fc_columns].clip(-1.4, upper=1.4)
combined_df_clipped

In [ ]:
## Plot all KO targets on a single scatter, colored by target
cmap = plt.get_cmap('tab10')
target_colors = {target: cmap(i % cmap.N) for i, target in enumerate(KO_genes)}

plt.figure(figsize=(17, 14))
texts = []

all_x, all_y = [], []

for target in KO_genes:
    sig_mask_filter = (
        (combined_df_clipped[f'{target}_pvalue'] < 0.05)
        & (combined_df_clipped[f'{target}_pvalue_LG'] < 0.05)
    )
    combined_signif_df = combined_df_clipped[sig_mask_filter]

    if combined_signif_df.empty:
        continue

    color = target_colors[target]

    x_vals = combined_signif_df[f'{target}_log2FC_LG']
    y_vals = combined_signif_df[f'{target}_log2FC']

    all_x.extend(x_vals.tolist())
    all_y.extend(y_vals.tolist())

    plt.scatter(x_vals, y_vals, color=color, alpha=0.6, s=12, label=target)

    for lipid, row in combined_signif_df.iterrows():
        texts.append(
            plt.text(
                row[f'{target}_log2FC_LG'],
                row[f'{target}_log2FC'],
                lipid,
                fontsize=20,
                color=color,
                ha='center',
                va='center',
            )
        )

# Pearson correlation across all significant points
rho, pval = pearsonr(all_x, all_y)
pval_str = f'{pval:.2e}' if pval < 1e-3 else f'{pval:.4f}'
corr_text = f'pearson r = {rho:.3f}\np = {pval_str}\nn = {len(all_x)}'

plt.axhline(y=0, color='red', linestyle='--')
plt.axvline(x=0, color='red', linestyle='--')
plt.xlabel('Log2FC (LipoGrid)')
plt.ylabel('Log2FC (Bulk)')
plt.grid()

# Smaller legend with target names colored to match the points
legend = plt.legend(
    title='Target',
    loc='best',
    fontsize=22,
    title_fontsize=18,
    markerscale=0,
    handlelength=0,
    borderpad=0.3,
    labelspacing=0.3,
    handletextpad=0,
    frameon=True,
)
for text in legend.get_texts():
    text.set_color(target_colors[text.get_text()])

# Annotate the correlation in the top-left corner
plt.gca().text(
    0.02, 0.98, corr_text,
    transform=plt.gca().transAxes,
    fontsize=24,
    va='top', ha='left',
    bbox=dict(boxstyle='round,pad=0.4', facecolor='white', edgecolor='gray', alpha=0.85),
)

plt.tight_layout()

adjust_text(
    texts,
    arrowprops=dict(arrowstyle='-', color='gray', lw=1),
    expand_text=(1.3, 1.3),
    expand_points=(1.2, 1.2),
    force_text=0.3,
    force_points=0.3,
    force_pull=7,
    time_lim=15,
    min_arrow_len=15,
)

plt.savefig(
    '../data/lipogrid/pilot/bulk_lipidomics/corrplot_bulk_MALDI_all_targets_log2FC_p0.01LG_0.1Bulk.pdf'
)
plt.show()